# Frozen-feature VGG19 classifier

This notebook follows the official PyTorch transfer-learning pattern, adapted to VGG19. The goal is to keep VGG19's pretrained visual layers fixed and train only a small final layer for a two-class ants/bees task.

Source pattern: <https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html>  
VGG19 reference: <https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vgg19.html>


## 1. Load pretrained VGG19 and freeze its feature extractor

VGG19 was pretrained on ImageNet, so its convolutional `features` layers already detect useful visual patterns such as edges, colors, textures, and shapes. Freezing means those stored weights still run during the forward pass, but PyTorch will not update them during training.

This first cell demonstrates the core freezing operation on the feature extractor.


In [7]:
from torchvision.models import VGG19_Weights, vgg19

# DEFAULT loads the ImageNet-pretrained VGG19 weights provided by torchvision.
weights = VGG19_Weights.DEFAULT
model = vgg19(weights=weights)

# Freeze only the convolutional feature extractor for this first inspection.
# Frozen parameters still process images, but training will not change their weights.
for parameter in model.features.parameters():
    parameter.requires_grad = False


## 2. Build the strict fixed-feature classifier setup

For the actual transfer-learning setup, we freeze every pretrained VGG19 parameter first. Then we replace the old ImageNet final layer with a new two-output layer for `ants` and `bees`.

The new layer is created after freezing, so it keeps the default `requires_grad=True` setting and becomes the only trainable part.


In [6]:
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

weights = VGG19_Weights.DEFAULT
model = vgg19(weights=weights)

# Freeze every pretrained VGG19 weight so the old ImageNet knowledge stays fixed.
for parameter in model.parameters():
    parameter.requires_grad = False

# The old final layer accepted this many input values from the previous layer.
# We reuse that input width so the replacement layer still fits into VGG19.
in_features = model.classifier[6].in_features

# Replace ImageNet's 1000-class final layer with a 2-class ants/bees layer.
# Because this layer is new, its weight and bias remain trainable.
model.classifier[6] = nn.Linear(in_features, 2)


## 3. Verify which parameters will train

Before training, inspect the trainable parameters. In this strict frozen-feature setup, the expected output is only:

```text
classifier.6.weight
classifier.6.bias
```

That check confirms the pretrained VGG19 body is frozen and only the replacement classifier layer will learn.


In [5]:
# Print the names of parameters PyTorch is allowed to update during training.
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name)


classifier.6.weight
classifier.6.bias
